# 13 - Comparação de modelos do P1: Isolation Forest × Autoencoder
**Squad 1 · Data Quality em Tempo Real | ML em Streaming**
**Responsável:** Riuler José Silvério · **Modelo B:** Guilherme Silva & Carlos Jerônimo
**Card do Trello:** Comparação de modelos

> Card: *"Comparar modelos que resolvem o mesmo problema, identificando quais features foram usadas de entrada do modelo, qual o tipo de dado o modelo de machine learning melhor se adequa, qual teve melhor performance, etc. Entregar uma one page com o resultado da comparação."*

### Modelos do P1 (anomalias em pedidos e itens)
| | Modelo A · Isolation Forest | Modelo B · Autoencoder denso |
|---|---|---|
| Dupla | Riuler | Guilherme & Carlos |
| Grão | **item** de pedido | **pedido** |
| Versões | baseline (`10`) e evolução (`12`) | a definir pela dupla 4 |
| Saída lida aqui | `squad1/dupla1/ml/scores_itens_pedido` e `..._v2` | `CAMINHO_SCORES_AE` (contrato abaixo) |

### Protocolo comum
Os dois modelos olham grãos diferentes, então a comparação é feita **nos dois níveis**:
- **Nível pedido (principal):** o pedido é anômalo no P1 se tem **algum item com regra quebrada** (as 4 regras de item) **ou** `dq_valor_total_invalido` **ou** `dq_frete_invalido` (as regras de **valor** da Silver de pedidos). O score do Isolation Forest no pedido é o **maior score entre os itens**.
- **Nível item:** o rótulo é o rótulo fraco do item, e o score do Autoencoder no pedido é **repetido em todos os itens** do pedido.
- **Teste:** os mesmos lotes de 08/10 e 15/10 (1.135 pedidos e 6.121 itens). O treino de ambos vai até 07/10.
- **Métricas:** Precision, Recall, F1, PR-AUC e % alertados, mais a **complementaridade**: quanto cada modelo pega que o outro não pega.

As outras regras da Silver de pedidos (status, datas e endereço) **não entram** no rótulo, porque não são anomalias de valor, que é o escopo do P1.

### Contrato para os scores do Autoencoder
Uma tabela Delta (ou CSV) em `CAMINHO_SCORES_AE`, com **uma linha por pedido** (pelo menos os do teste):

| coluna | tipo | descrição |
|---|---|---|
| `id_pedido` | long | chave do pedido |
| `score_anomalia` | double | erro de reconstrução, em que **maior = mais anômalo** |
| `alerta` | int | 1 se passou do limiar (percentil 99 do treino, como na planilha) |
| `versao_modelo` | string | ex.: `autoencoder v1 · 32-16-8 · p99` |

Enquanto a tabela não existir, o notebook roda só com o Isolation Forest e mostra o lado do Autoencoder como **aguardando**.

In [ ]:
# Credenciais via .env (nunca no código). O .env fica fora do Git (.gitignore).
import os
from dotenv import load_dotenv

CAMINHOS_ENV = [".env", "../.env", "../../estagio-empregadados-turma-2/.env"]
caminho_env = next((p for p in CAMINHOS_ENV if os.path.exists(p)), None)
if caminho_env:
    load_dotenv(caminho_env)

client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")
assert all([client_id, tenant_id, client_secret, storage_account]), "Credenciais do ADLS não encontradas no .env"

adls_options = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}
base_squad1 = f"abfss://squad1@{storage_account}.dfs.core.windows.net"
caminhos = {
    "if_baseline": f"{base_squad1}/dupla1/ml/scores_itens_pedido",        # notebook 10
    "if_evolucao": f"{base_squad1}/dupla1/ml/scores_itens_pedido_v2",     # notebook 12
    "silver_pedidos": f"{base_squad1}/silver/ecommerce_pedidos",          # dupla 4, somente leitura
}
# Ajustar quando a dupla 4 informar o caminho dos scores do Autoencoder (Delta). None = ainda não existe.
CAMINHO_SCORES_AE = None
print("Credenciais carregadas do .env (somente leitura).")

### Etapa 1 — Leitura

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ler = lambda p: spark.read.format("delta").options(**adls_options).load(p).toPandas()
if_base, if_evo = ler(caminhos["if_baseline"]), ler(caminhos["if_evolucao"])
pedidos = ler(caminhos["silver_pedidos"])
ae = None
if CAMINHO_SCORES_AE:
    try:
        ae = ler(CAMINHO_SCORES_AE)
    except Exception as e:
        print(f"Scores do Autoencoder não encontrados em {CAMINHO_SCORES_AE}: {type(e).__name__}")

In [ ]:
AZUL, VERMELHO, AMARELO, CINZA, VERDE = "#1F4E8C", "#C0392B", "#F2B705", "#7F8C8D", "#2E7D32"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False})
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, precision_recall_curve

TEM_AE = ae is not None and len(ae) > 0
if TEM_AE:
    faltando = {"id_pedido", "score_anomalia", "alerta"} - set(ae.columns)
    assert not faltando, f"Scores do Autoencoder sem as colunas do contrato: {faltando}"

# Itens do teste com os scores das duas versões do Isolation Forest
chaves = ["id_item_pedido", "id_pedido", "sku", "lote", "data_lote", "split", "rotulo_fraco"]
itens = (if_base[if_base.split == "teste"][chaves + ["score_anomalia", "alerta"]]
         .rename(columns={"score_anomalia": "score_if_baseline", "alerta": "alerta_if_baseline"})
         .merge(if_evo[["id_item_pedido", "score_anomalia", "alerta"]]
                .rename(columns={"score_anomalia": "score_if_evolucao", "alerta": "alerta_if_evolucao"}), on="id_item_pedido"))

# Rótulo de valor do pedido (regras da Silver de pedidos da dupla 4)
pedidos = pedidos.drop_duplicates("id_pedido")
pedidos["rotulo_valor_pedido"] = (pedidos.dq_valor_total_invalido | pedidos.dq_frete_invalido).astype(int)

# Nível pedido: o Isolation Forest usa o maior score entre os itens do pedido
ped = (itens.groupby("id_pedido").agg(n_itens=("id_item_pedido", "size"), rotulo_itens=("rotulo_fraco", "max"),
                                      score_if_baseline=("score_if_baseline", "max"), alerta_if_baseline=("alerta_if_baseline", "max"),
                                      score_if_evolucao=("score_if_evolucao", "max"), alerta_if_evolucao=("alerta_if_evolucao", "max"))
       .reset_index()
       .merge(pedidos[["id_pedido", "valor_total", "valor_frete", "metodo_pagamento", "rotulo_valor_pedido",
                       "dq_valor_total_invalido", "dq_frete_invalido"]], on="id_pedido", how="left"))
ped["rotulo_p1"] = ((ped.rotulo_itens == 1) | (ped.rotulo_valor_pedido == 1)).astype(int)

MODELOS = {"IF baseline": "if_baseline", "IF evolução": "if_evolucao"}
if TEM_AE:
    ae_cols = ae[["id_pedido", "score_anomalia", "alerta"]].rename(columns={"score_anomalia": "score_ae", "alerta": "alerta_ae"})
    ped = ped.merge(ae_cols, on="id_pedido", how="left")
    itens = itens.merge(ae_cols, on="id_pedido", how="left")      # nível item: score do pedido repetido nos itens
    MODELOS["Autoencoder"] = "ae"
    print(f"Pedidos do teste sem score do Autoencoder: {ped.score_ae.isna().sum()}")

print(f"Teste: {len(ped):,} pedidos e {len(itens):,} itens | Autoencoder disponível: {TEM_AE}")
print(f"Rótulos no nível pedido: P1 {ped.rotulo_p1.mean():.1%} | item com regra {ped.rotulo_itens.mean():.1%} | "
      f"valor total/frete {ped.rotulo_valor_pedido.mean():.1%}")

### Etapa 2 — Ficha técnica
O card pede as features de entrada, o tipo de dado mais adequado a cada modelo e o desempenho. A ficha abaixo cobre os dois primeiros pontos; o lado do Autoencoder é preenchido com a dupla 4.

| | IF baseline | IF evolução | Autoencoder |
|---|---|---|---|
| Tipo | não supervisionado, árvores de isolamento | não supervisionado, árvores de isolamento | não supervisionado, rede neural (erro de reconstrução) |
| Grão | item | item | pedido |
| Features | preço (log), quantidade, desconto ÷ preço, valor líquido do item, nº de itens, nº de SKUs e valor do pedido (7) | preço ÷ típico do SKU, quantidade, desconto ÷ preço, valor líquido do item (4) | features do pedido normalizadas + `valor_total`, `valor_frete`, método de pagamento (planilha) |
| Pré-processamento | log com sinal; sem normalização | log com sinal; referência de preço por SKU | normalização obrigatória; one-hot do método de pagamento |
| Limiar | contamination 0,09 | contamination 0,08 (F1 na validação) | percentil 99 do treino (planilha) |
| Melhor tipo de dado | tabular pequeno, poucas features, valores **extremos** | idem, com contexto do produto | tabular com **relações entre colunas** (ex.: total × itens × frete) e mais volume |
| Interpretação | alta: dá para apontar a feature mais distante | alta: "× típico do SKU" | média: erro de reconstrução por feature |
| Custo | segundos em CPU, sem tuning sensível | idem | maior; sensível a arquitetura e normalização |

### Etapa 3 — Desempenho no nível pedido (principal)

In [ ]:
def metricas(y, score, alerta):
    ok = score.notna()
    y, score, alerta = y[ok], score[ok], alerta[ok]
    return {"% alertados": alerta.mean() * 100, "Precision": precision_score(y, alerta, zero_division=0),
            "Recall": recall_score(y, alerta, zero_division=0), "F1": f1_score(y, alerta, zero_division=0),
            "PR-AUC": average_precision_score(y, score)}

def tabela(base, rotulos):
    linhas = []
    for nome_rot, col_rot in rotulos.items():
        for nome, sufixo in MODELOS.items():
            linhas.append({"rótulo": nome_rot, "modelo": nome, **metricas(base[col_rot], base[f"score_{sufixo}"], base[f"alerta_{sufixo}"])})
        linhas.append({"rótulo": nome_rot, "modelo": "aleatório", "PR-AUC": base[col_rot].mean()})
    return pd.DataFrame(linhas)

ROT_PEDIDO = {"P1 (item com regra ou valor/frete)": "rotulo_p1", "só item com regra": "rotulo_itens", "só valor total/frete": "rotulo_valor_pedido"}
res_pedido = tabela(ped, ROT_PEDIDO)
display(res_pedido.round(3))
if not TEM_AE:
    print("Autoencoder: aguardando os scores da dupla 4.")

### Etapa 4 — Desempenho no nível item

In [ ]:
res_item = tabela(itens, {"rótulo fraco do item": "rotulo_fraco"})
display(res_item.round(3))

### Etapa 5 — Complementaridade (nível pedido, rótulo P1)
Mostra, entre os pedidos anômalos, quantos cada modelo pega **sozinho**, quantos os dois pegam e quanto a **união** dos alertas cobriria. Se a união ganha muito Recall, os modelos são complementares e podem rodar juntos.

In [ ]:
def complementaridade(a, b, nome_a, nome_b):
    y = ped.rotulo_p1 == 1
    A, B = ped[f"alerta_{a}"] == 1, ped[f"alerta_{b}"] == 1
    return {"par": f"{nome_a} × {nome_b}", f"só {nome_a}": int((y & A & ~B).sum()), f"só {nome_b}": int((y & B & ~A).sum()),
            "os dois": int((y & A & B).sum()), "nenhum": int((y & ~A & ~B).sum()),
            f"Recall {nome_a}": (y & A).sum() / y.sum(), f"Recall {nome_b}": (y & B).sum() / y.sum(), "Recall união": (y & (A | B)).sum() / y.sum(),
            "% pedidos alertados (união)": (A | B).mean() * 100}

pares = [("if_baseline", "if_evolucao", "IF baseline", "IF evolução")]
if TEM_AE:
    pares += [("if_evolucao", "ae", "IF evolução", "Autoencoder"), ("if_baseline", "ae", "IF baseline", "Autoencoder")]
for a, b, na, nb in pares:
    display(pd.DataFrame([complementaridade(a, b, na, nb)]).round(3))

# Recall por tipo de erro no nível pedido
tipos = {"item com regra": ped.rotulo_itens == 1, "valor total inválido": ped.dq_valor_total_invalido == True, "frete inválido": ped.dq_frete_invalido == True}
display(pd.DataFrame([{"tipo de erro": k, "pedidos": int(m.sum()), **{f"Recall {n}": ped.loc[m, f"alerta_{s}"].mean() for n, s in MODELOS.items()}}
                      for k, m in tipos.items()]).round(3))

### Etapa 6 — Curvas Precision × Recall (nível pedido, rótulo P1)

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 4.5))
cores = {"IF baseline": CINZA, "IF evolução": AZUL, "Autoencoder": AMARELO}
for ax, (titulo, rot) in zip(eixos, [("P1 (item com regra ou valor/frete)", "rotulo_p1"), ("só valor total/frete", "rotulo_valor_pedido")]):
    for nome, s in MODELOS.items():
        ok = ped[f"score_{s}"].notna()
        p, r, _ = precision_recall_curve(ped.loc[ok, rot], ped.loc[ok, f"score_{s}"])
        ax.plot(r, p, lw=2, color=cores[nome], label=f"{nome} ({average_precision_score(ped.loc[ok, rot], ped.loc[ok, f'score_{s}']):.2f})")
    ax.axhline(ped[rot].mean(), color=VERMELHO, ls="--", lw=1, label="aleatório")
    ax.set_title(titulo)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### Etapa 7 — Conclusões (parciais: aguardando o Autoencoder)

> Execução de 22/11 com o teste de 08/10 e 15/10 (1.135 pedidos e 6.121 itens). **O Autoencoder ainda não tem scores**; as conclusões abaixo cobrem o Modelo A e o protocolo. Quando a dupla 4 informar `CAMINHO_SCORES_AE`, basta rodar de novo.

**Nível pedido (principal)**
| Rótulo | Pedidos | IF baseline PR-AUC | IF evolução PR-AUC | Autoencoder | Aleatório |
|---|---|---|---|---|---|
| P1 (item com regra ou valor/frete) | 35,6% | 0,844 | **0,879** | aguardando | 0,356 |
| Só valor total/frete | 2,7% (31) | **0,381** | 0,097 | aguardando | 0,027 |

**Nível item:** a PR-AUC é 0,741 na baseline e 0,808 na evolução (aleatório: 0,083).

**O que já dá para concluir**
1. **Para erros de item, a evolução é o melhor Modelo A**, nos dois níveis (pedido: 0,879; item: 0,808).
2. **Erros de valor do pedido são o ponto fraco do Modelo A.** A baseline, que tem features de pedido, ordena esses pedidos razoavelmente (0,38) e pega os 15 de valor total inválido. A evolução, que só olha o item, fica perto do acaso (0,10). Como o Autoencoder usa `valor_total`, `valor_frete` e método de pagamento, a expectativa é que ele seja **mais forte justamente aqui**, e por isso a comparação mede a complementaridade, e não só quem vence.
3. **Baseline e evolução se complementam pouco** no rótulo P1: a união dos alertas leva o Recall de 0,772 para 0,797, mas aumenta os pedidos alertados para 36,7%.
4. **No nível pedido, o % alertado é alto (29% a 34%)**, porque basta um item alertado para alertar o pedido, e 35,6% dos pedidos do teste têm algum erro. Para comparar com o Autoencoder (limiar no percentil 99), a PR-AUC, que não depende do limiar, é a métrica mais justa.

**O que falta**
- Scores do Autoencoder no contrato descrito no início, com o mesmo treino (até 07/10) e o mesmo teste.
- A ficha técnica do Autoencoder: arquitetura, features finais, normalização e limiar.
- Rodar este notebook de novo e fechar o one page.

**Limitações:** há só 31 pedidos com erro de valor total ou frete no teste, então as métricas desse rótulo variam muito. O rótulo fraco não é a verdade absoluta.
